# 01 - Exploratory Data Analysis

Multi-Region Marketing Mix Modeling Dataset

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import re
import warnings
import missingno as msno
import holidays
from statsmodels.tsa.seasonal import seasonal_decompose

from src.config import RAW_DATA_DIR, DATA_FILENAME, DICT_FILENAME

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 50)

## 1. Load Data

In [ ]:
# Load main dataset
df = pd.read_csv(f"{RAW_DATA_DIR}/{DATA_FILENAME}")

print(f"Shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
# Load data dictionary
data_dict = pd.read_excel(f"{RAW_DATA_DIR}/{DICT_FILENAME}")
data_dict

In [ ]:
df.head()

In [ ]:
df.info()

## 2. Data Overview

In [ ]:
# Date range
df['DATE_DAY'] = pd.to_datetime(df['DATE_DAY'])
print(f"Date range: {df['DATE_DAY'].min()} to {df['DATE_DAY'].max()}")
print(f"Duration: {(df['DATE_DAY'].max() - df['DATE_DAY'].min()).days} days")

In [ ]:
# Unique values for categorical columns
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"{col}: {df[col].nunique()} unique values")

In [ ]:
# Organizations and regions
print(f"Unique organisations: {df['ORGANISATION_ID'].nunique()}")
print(f"Unique regions: {df['ORGANISATION_PRIMARY_TERRITORY_NAME'].nunique()}")
print(f"\nRegions: {df['ORGANISATION_PRIMARY_TERRITORY_NAME'].unique()}")

## 3. Missing Data Analysis

In [ ]:
# Missing values percentage
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_df = missing[missing > 0].reset_index()
missing_df.columns = ['Column', 'Missing %']

In [ ]:
missing_df

### 3.1 Missing Data Patterns (Heatmap)

In [ ]:
# Nullity heatmap - shows correlation of missing values between columns
msno.heatmap(df, figsize=(12, 8), fontsize=10)
plt.title('Missing Values Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Nullity matrix - shows distribution of missing values across rows
msno.matrix(df, figsize=(14, 6), fontsize=10, sparkline=True)
plt.title('Missing Data Matrix')
plt.tight_layout()
plt.show()

## 4. Channel Analysis

In [ ]:
# A análise de gastos (spend) precisa levar em consideração alguns fatores:
# 1. O tempo. Receita é fluxo, não estoque. ('DATE_DAY')
# 2. O território, a moeda local. ('ORGANISATION_PRIMARY_TERRITORY_NAME','TERRITORY_NAME', 'CURRENCY_CODE')

In [ ]:
# Identify spend columns
spend_cols = [c for c in df.columns if 'SPEND' in c]
click_cols = [c for c in df.columns if 'CLICK' in c]
impression_cols = [c for c in df.columns if 'IMPRESSION' in c]

print(f"Spend columns ({len(spend_cols)}): {spend_cols}")
print(f"Click columns ({len(click_cols)}): {click_cols}")
print(f"Impression columns ({len(impression_cols)}): {impression_cols}")

In [ ]:
# Filter only columns with channel suffixes
channel_pattern = r'_(SPEND|CLICKS|IMPRESSIONS)$'
missing_channels = missing_df[missing_df['Column'].str.contains(channel_pattern, regex=True)].copy()

# Extract channel name
missing_channels['Channel'] = missing_channels['Column'].str.replace(channel_pattern, '', regex=True)

# Group by channel
missing_df_grouped = missing_channels.groupby('Channel').agg({'Missing %': 'mean'}).reset_index()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=missing_df_grouped,
    x='Missing %',
    y='Channel', 
    ax=ax,
    order=missing_df_grouped.sort_values('Missing %', ascending=False)['Channel']
)
ax.bar_label(ax.containers[0], fmt='%.1f', padding=-30, label_type='edge', color='white', fontweight='bold')
ax.set_title('Channels with Missing Data')
ax.set_xlabel('Missing Percentage (%)')
plt.tight_layout()
plt.show()

In [ ]:
# Total spend per channel
spend_totals = pd.DataFrame(df[spend_cols].sum().sort_values(ascending=False), columns=['Total Spend'])
spend_totals = (spend_totals
    .reset_index()
    .rename(columns={'index': 'Channel'}))
spend_totals['Total Spend (in thousands)'] = spend_totals['Total Spend'] / 1e3

spend_totals['Channel'] = spend_totals['Channel'].str.replace('_SPEND', '', regex=True)

In [ ]:
# Visualize total spend per channel
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=spend_totals,
    x='Total Spend (in thousands)',
    y='Channel',
    ax=ax,
    order=spend_totals.sort_values('Total Spend (in thousands)', ascending=False)['Channel'],
)
ax.set_title('Total Spend by Channel')
ax.set_xlabel('Total Spend (in thousands)')
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x:,.0f}'))
ax.bar_label(
    ax.containers[0], 
    fmt=lambda x: f'$ {x:,.0f}',
    label_type='edge', 
    color='grey', 
    fontweight='bold'
)
ax.set_xlim(0, 800*1e3)
plt.tight_layout()
plt.show()

### 4.1 Impressions & Clicks by Channel

In [ ]:
# Total impressions per channel
if impression_cols:
    impression_totals = pd.DataFrame(df[impression_cols].sum().sort_values(ascending=False), columns=['Total Impressions'])
    impression_totals = impression_totals.reset_index().rename(columns={'index': 'Channel'})
    impression_totals['Impressions (millions)'] = impression_totals['Total Impressions'] / 1e6
    impression_totals['Channel'] = impression_totals['Channel'].str.replace('_IMPRESSIONS', '', regex=True)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(
        data=impression_totals,
        x='Impressions (millions)',
        y='Channel',
        ax=ax,
        order=impression_totals.sort_values('Impressions (millions)', ascending=False)['Channel'],
        palette='Blues_d'
    )
    ax.set_title('Total Impressions by Channel')
    ax.set_xlabel('Impressions (millions)')
    ax.bar_label(ax.containers[0], fmt=lambda x: f'{x:,.1f}M', label_type='edge', color='grey', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Total clicks per channel
if click_cols:
    click_totals = pd.DataFrame(df[click_cols].sum().sort_values(ascending=False), columns=['Total Clicks'])
    click_totals = click_totals.reset_index().rename(columns={'index': 'Channel'})
    click_totals['Clicks (thousands)'] = click_totals['Total Clicks'] / 1e3
    click_totals['Channel'] = click_totals['Channel'].str.replace('_CLICKS', '', regex=True)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(
        data=click_totals,
        x='Clicks (thousands)',
        y='Channel',
        ax=ax,
        order=click_totals.sort_values('Clicks (thousands)', ascending=False)['Channel'],
        palette='Greens_d'
    )
    ax.set_title('Total Clicks by Channel')
    ax.set_xlabel('Clicks (thousands)')
    ax.bar_label(ax.containers[0], fmt=lambda x: f'{x:,.0f}K', label_type='edge', color='grey', fontweight='bold')
    plt.tight_layout()
    plt.show()

### 4.2 Channel Efficiency Metrics (CTR & CPC)

In [ ]:
# Build channel metrics dataframe
channel_names = [c.replace('_SPEND', '') for c in spend_cols]

channel_metrics = pd.DataFrame({
    'Channel': channel_names,
    'Total Spend': [df[f'{ch}_SPEND'].sum() for ch in channel_names],
    'Total Clicks': [df[f'{ch}_CLICKS'].sum() if f'{ch}_CLICKS' in df.columns else 0 for ch in channel_names],
    'Total Impressions': [df[f'{ch}_IMPRESSIONS'].sum() if f'{ch}_IMPRESSIONS' in df.columns else 0 for ch in channel_names]
})

# Calculate CTR and CPC
channel_metrics['CTR (%)'] = (channel_metrics['Total Clicks'] / channel_metrics['Total Impressions'] * 100).replace([np.inf, -np.inf], np.nan)
channel_metrics['CPC ($)'] = (channel_metrics['Total Spend'] / channel_metrics['Total Clicks']).replace([np.inf, -np.inf], np.nan)

channel_metrics.style.format({
    'Total Spend': '$ {:,.0f}',
    'Total Clicks': '{:,.0f}',
    'Total Impressions': '{:,.0f}',
    'CTR (%)': '{:.2f}%',
    'CPC ($)': '${:.2f}'
})

In [ ]:
# Visualize CTR by Channel
ctr_data = channel_metrics[channel_metrics['CTR (%)'].notna()].sort_values('CTR (%)', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=ctr_data,
    x='CTR (%)',
    y='Channel',
    ax=ax,
    palette='Oranges_d'
)
ax.set_title('Click-Through Rate (CTR) by Channel')
ax.set_xlabel('CTR (%)')
ax.bar_label(ax.containers[0], fmt=lambda x: f'{x:.2f}%', label_type='edge', color='grey', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize CPC by Channel
cpc_data = channel_metrics[channel_metrics['CPC ($)'].notna()].sort_values('CPC ($)', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=cpc_data,
    x='CPC ($)',
    y='Channel',
    ax=ax,
    palette='Reds_d'
)
ax.set_title('Cost Per Click (CPC) by Channel')
ax.set_xlabel('CPC ($)')
ax.bar_label(ax.containers[0], fmt=lambda x: f'${x:.2f}', label_type='edge', color='grey', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Channel Efficiency Metrics (CTR & CPC) over time

In [ ]:
channel_names

In [ ]:
# Line plot of all channels' CTR over time
fig, ax = plt.subplots(figsize=(12, 5))
colors = sns.color_palette("Set1", n_colors=len(channel_names))
window = 21  # Janela de média móvel

for i, channel in enumerate(channel_names):
    clicks_col = f'{channel}_CLICKS'
    impressions_col = f'{channel}_IMPRESSIONS'
    
    if clicks_col in df.columns and impressions_col in df.columns:
        ctr_data_time = (
            df.groupby('DATE_DAY')
            .agg({clicks_col: 'sum', impressions_col: 'sum'})
            .assign(CTR=lambda x: x[clicks_col] / x[impressions_col] * 100)
            .sort_index()
        )
        ctr_smoothed = ctr_data_time['CTR'].rolling(window=window, min_periods=1).mean()
        ax.plot(ctr_data_time.index, ctr_smoothed, label=channel, color=colors[i])

ax.set_title(f'CTR Over Time by Channel ({window}-day Moving Average)')
ax.set_xlabel('Date')
ax.set_ylabel('CTR (%)')
ax.legend(title='Channel', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

In [ ]:
# Line plot of all channels' CTR over time, except GOOGLE_PAID_SEARCH
# Rolling window of 14 days
fig, ax = plt.subplots(figsize=(12, 5))
colors = sns.color_palette("Set1", n_colors=len(channel_names))
window = 21  # Janela de média móvel

for i, channel in enumerate([channel for channel in channel_names if 'GOOGLE_PAID_SEARCH' not in channel]):
    clicks_col = f'{channel}_CLICKS'
    impressions_col = f'{channel}_IMPRESSIONS'
    
    if clicks_col in df.columns and impressions_col in df.columns:
        ctr_data_time = (
            df.groupby('DATE_DAY')
            .agg({clicks_col: 'sum', impressions_col: 'sum'})
            .assign(CTR=lambda x: x[clicks_col] / x[impressions_col] * 100)
            .sort_index()
        )
        # Média móvel
        ctr_smoothed = ctr_data_time['CTR'].rolling(window=window, min_periods=1).mean()
        ax.plot(ctr_data_time.index, ctr_smoothed, label=channel, color=colors[i])

ax.set_title(f'CTR Over Time by Channel ({window}-day Moving Average)')
ax.set_xlabel('Date')
ax.set_ylabel('CTR (%)')
ax.legend(title='Channel', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## 5. Revenue Analysis

In [ ]:
# Revenue analysis context:
# 1. Revenue is a flow, not stock (DATE_DAY)
# 2. Currency and region matter (ORGANISATION_PRIMARY_TERRITORY_NAME, CURRENCY_CODE)

In [ ]:
# Identify revenue-related columns
revenue_cols = [c for c in df.columns if 'PURCHASE' in c or 'REVENUE' in c or 'PRICE' in c]
print(f"Revenue columns: {revenue_cols}")

In [ ]:
df['revenue'] = df['ALL_PURCHASES_ORIGINAL_PRICE'] - df['ALL_PURCHASES_GROSS_DISCOUNT']
print(f"Revenue stats:\n")
df['revenue'].describe().apply(lambda x: f'{x:,.0f}')

In [ ]:
# Revenue distribution
if 'revenue' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    df['revenue'].hist(bins=50, ax=axes[0])
    axes[0].set_title('Revenue Distribution')
    axes[0].set_xlabel('Revenue')
    
    np.log1p(df['revenue']).hist(bins=50, ax=axes[1])
    axes[1].set_title('Log Revenue Distribution')
    axes[1].set_xlabel('Log(Revenue + 1)')
    
    plt.tight_layout()
    plt.show()

### 5.1 Revenue Outlier Detection

In [ ]:
# Boxplot for revenue outlier detection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall revenue boxplot
axes[0].boxplot(df['revenue'].dropna(), vert=True)
axes[0].set_title('Revenue Distribution (Boxplot)')
axes[0].set_ylabel('Revenue')
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x/1e3:.0f}K'))

# Log-transformed boxplot
axes[1].boxplot(np.log1p(df['revenue'].dropna()), vert=True)
axes[1].set_title('Log(Revenue + 1) Distribution (Boxplot)')
axes[1].set_ylabel('Log(Revenue + 1)')

plt.tight_layout()
plt.show()

In [ ]:
# Revenue boxplot by region
region_col = 'ORGANISATION_PRIMARY_TERRITORY_NAME'
fig, ax = plt.subplots(figsize=(14, 6))

region_order = df.groupby(region_col)['revenue'].median().sort_values(ascending=False).index
df.boxplot(column='revenue', by=region_col, ax=ax, positions=range(len(region_order)))
ax.set_xticklabels(region_order, rotation=45, ha='right')
ax.set_title('Revenue Distribution by Region')
ax.set_xlabel('Region')
ax.set_ylabel('Revenue')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x/1e3:.0f}K'))
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Outlier statistics using IQR method
Q1 = df['revenue'].quantile(0.25)
Q3 = df['revenue'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['revenue'] < lower_bound) | (df['revenue'] > upper_bound)]
print(f"IQR-based outlier analysis:")
print(f"  Q1: ${Q1:,.0f}")
print(f"  Q3: ${Q3:,.0f}")
print(f"  IQR: ${IQR:,.0f}")
print(f"  Lower bound: ${lower_bound:,.0f}")
print(f"  Upper bound: ${upper_bound:,.0f}")
print(f"  Outlier count: {len(outliers):,} ({len(outliers)/len(df)*100:.2f}%)")

### 5.2 Revenue by Organization

In [ ]:
# Top organizations by revenue
if 'revenue' in df.columns:
    org_revenue = df.groupby('ORGANISATION_ID')['revenue'].sum().sort_values(ascending=False)
    top_n = 15
    
    fig, ax = plt.subplots(figsize=(12, 6))
    org_revenue.head(top_n).plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'Top {top_n} Organizations by Total Revenue')
    ax.set_xlabel('Organization ID')
    ax.set_ylabel('Total Revenue')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    # Revenue concentration
    top_5_share = org_revenue.head(5).sum() / org_revenue.sum() * 100
    top_10_share = org_revenue.head(10).sum() / org_revenue.sum() * 100
    print(f"\nRevenue concentration:")
    print(f"  Top 5 organizations: {top_5_share:.1f}% of total revenue")
    print(f"  Top 10 organizations: {top_10_share:.1f}% of total revenue")

## 6. Temporal Patterns

In [ ]:
# Daily aggregation
daily = df.groupby('DATE_DAY').agg({
    'revenue': 'sum' if 'revenue' in df.columns else 'count',
    spend_cols[0]: 'sum' if spend_cols else 'count'
}).reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily['DATE_DAY'], daily['revenue'] if 'revenue' in daily.columns else daily.iloc[:, 1])
ax.set_title('Daily Revenue Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue')
plt.tight_layout()
plt.show()

In [ ]:
# Day of week pattern
df['day_of_week'] = df['DATE_DAY'].dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

if 'revenue' in df.columns:
    dow_revenue = df.groupby('day_of_week')['revenue'].mean().reindex(dow_order)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    dow_revenue.plot(kind='bar', ax=ax, color='coral')
    ax.set_title('Average Revenue by Day of Week')
    ax.set_ylabel('Mean Revenue')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
    plt.tight_layout()
    plt.show()

### 6.1 Monthly Revenue Trend

In [ ]:
# Monthly aggregation
if 'revenue' in df.columns:
    df['year_month'] = df['DATE_DAY'].dt.to_period('M')
    monthly = df.groupby('year_month').agg({
        'revenue': 'sum',
        spend_cols[0]: 'sum' if spend_cols else 'count'
    }).reset_index()
    monthly['year_month'] = monthly['year_month'].astype(str)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(monthly['year_month'], monthly['revenue'], color='teal', alpha=0.8)
    ax.set_title('Monthly Revenue')
    ax.set_xlabel('Month')
    ax.set_ylabel('Total Revenue')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### 6.2 Time Series Decomposition

In [ ]:
# Time series decomposition to identify trend, seasonality, and residuals
daily_revenue = df.groupby('DATE_DAY')['revenue'].sum().sort_index()
daily_revenue.index = pd.DatetimeIndex(daily_revenue.index)

# Fill any missing dates
date_range = pd.date_range(start=daily_revenue.index.min(), end=daily_revenue.index.max(), freq='D')
daily_revenue = daily_revenue.reindex(date_range, fill_value=0)

# Decomposition (period=7 for weekly seasonality)
decomposition = seasonal_decompose(daily_revenue, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))

decomposition.observed.plot(ax=axes[0], title='Observed', color='steelblue')
axes[0].set_ylabel('Revenue')

decomposition.trend.plot(ax=axes[1], title='Trend', color='orange')
axes[1].set_ylabel('Revenue')

decomposition.seasonal.plot(ax=axes[2], title='Seasonal (Weekly)', color='green')
axes[2].set_ylabel('Revenue')

decomposition.resid.plot(ax=axes[3], title='Residual', color='red')
axes[3].set_ylabel('Revenue')

for ax in axes:
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
    ax.set_xlabel('')

plt.suptitle('Time Series Decomposition of Daily Revenue', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 6.3 Holiday Analysis

In [ ]:
# Build combined holidays for all regions
REGION_TO_COUNTRY = {
    'US': 'US', 'UK': 'GB', 'Australia': 'AU', 'Netherlands': 'NL',
    'Spain': 'ES', 'Hong Kong': 'HK', 'Ireland': 'IE', 'Canada': 'CA',
    'New Zealand': 'NZ', 'Germany': 'DE', 'Austria': 'AT', 'Japan': 'JP',
    'France': 'FR', 'Italy': 'IT', 'Sweden': 'SE', 'Denmark': 'DK',
    'Norway': 'NO', 'Switzerland': 'CH'
}

# Get date range from data
start_year = df['DATE_DAY'].min().year
end_year = df['DATE_DAY'].max().year

# Collect all holidays
all_holidays = set()
for region, country_code in REGION_TO_COUNTRY.items():
    try:
        country_holidays = holidays.country_holidays(country_code, years=range(start_year, end_year + 1))
        all_holidays.update(country_holidays.keys())
    except Exception:
        continue

In [ ]:
# Mark holidays in dataset
df['is_holiday'] = df['DATE_DAY'].dt.date.isin(all_holidays)
daily['is_holiday'] = daily['DATE_DAY'].dt.date.isin(all_holidays)

print(f"Total dates: {len(daily)}")
print(f"Holiday dates: {daily['is_holiday'].sum()}")
print(f"Holiday percentage: {daily['is_holiday'].sum() / len(daily) * 100:.1f}%")

In [ ]:
# Revenue time series with holidays marked
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(daily['DATE_DAY'], daily['revenue'], label='Daily Revenue', alpha=0.7)

# Mark holidays
holiday_dates = daily[daily['is_holiday']]['DATE_DAY']
holiday_revenue = daily[daily['is_holiday']]['revenue']
ax.scatter(holiday_dates, holiday_revenue, color='red', s=30, label='Holidays', zorder=5)

ax.set_title('Daily Revenue with Holidays Marked')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Holiday vs Non-Holiday revenue comparison
holiday_stats = daily.groupby('is_holiday')['revenue'].agg(['mean', 'median', 'std', 'count'])
holiday_stats.index = ['Non-Holiday', 'Holiday']
print("Revenue Statistics: Holiday vs Non-Holiday")
display(holiday_stats.style.format({
    'mean': '${:,.0f}',
    'median': '${:,.0f}',
    'std': '${:,.0f}',
    'count': '{:,.0f}'
}))

In [ ]:
# Boxplot: Holiday vs Non-Holiday revenue
fig, ax = plt.subplots(figsize=(8, 5))
daily.boxplot(column='revenue', by='is_holiday', ax=ax)
ax.set_xticklabels(['Non-Holiday', 'Holiday'])
ax.set_title('Revenue Distribution: Holidays vs Non-Holidays')
ax.set_xlabel('')
ax.set_ylabel('Revenue')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
plt.suptitle('')
plt.tight_layout()
plt.show()

## 7. Regional Analysis

In [ ]:
# Use correct column name for region
region_col = 'ORGANISATION_PRIMARY_TERRITORY_NAME'

# Revenue by region
if 'revenue' in df.columns and region_col in df.columns:
    region_stats = df.groupby(region_col).agg({
        'revenue': ['sum', 'mean', 'count']
    }).round(2)
    region_stats.columns = ['Total Revenue', 'Mean Revenue', 'Count']
    display(region_stats.sort_values('Total Revenue', ascending=False))

In [ ]:
df[df['ORGANISATION_PRIMARY_TERRITORY_NAME'] == 'Chile']['TERRITORY_NAME'].unique()

In [ ]:
# Spend by region
if region_col in df.columns and spend_cols:
    region_spend = df.groupby(region_col)[spend_cols].sum()
    region_spend.columns = [c.replace('_SPEND', '') for c in region_spend.columns]
    
    fig, ax = plt.subplots(figsize=(14, 6))
    region_spend.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title('Total Spend by Region and Channel')
    ax.set_xlabel('Region')
    ax.set_ylabel('Total Spend')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x/1e3:.0f}K'))
    ax.legend(title='Channel', bbox_to_anchor=(1.02, 1))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Revenue by region visualization
if 'revenue' in df.columns and region_col in df.columns:
    region_revenue = df.groupby(region_col)['revenue'].sum().sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    region_revenue.plot(kind='bar', ax=ax, color='darkgreen')
    ax.set_title('Total Revenue by Region')
    ax.set_xlabel('Region')
    ax.set_ylabel('Total Revenue')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 8. Vertical Analysis

In [ ]:
# Organization verticals
if 'ORGANISATION_VERTICAL' in df.columns:
    vertical_counts = df['ORGANISATION_VERTICAL'].value_counts()
    print(f"Verticals: {vertical_counts.to_dict()}")

In [ ]:
# Concentration Index (HHI) per vertical
def concentration_index(df):
    """Calculate HHI for each ORGANISATION_VERTICAL."""
    shares = (
        df.groupby('ORGANISATION_VERTICAL')['ORGANISATION_SUBVERTICAL']
        .value_counts(normalize=True)
    )
    hhi = shares.pow(2).groupby(level=0).sum().round(3)
    return hhi

if 'ORGANISATION_VERTICAL' in df.columns and 'ORGANISATION_SUBVERTICAL' in df.columns:
    hhi = concentration_index(df)
    print("HHI by Vertical (1.0 = monopoly, 0 = highly fragmented):")
    print(hhi.sort_values(ascending=False))

## 9. Correlation Analysis

In [ ]:
# Correlation between spend and revenue
if 'revenue' in df.columns and spend_cols:
    corr_cols = spend_cols + ['revenue']
    corr_matrix = df[corr_cols].corr()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=ax, fmt='.2f')
    ax.set_title('Correlation: Spend vs Revenue')
    plt.tight_layout()
    plt.show()

### 9.1 Bivariate Relationships (Pairplot)

In [ ]:
# Pairplot of top spend channels vs revenue
# Select top 3 spend channels by total spend
top_channels = spend_totals.nlargest(3, 'Total Spend')['Channel'].tolist()
top_spend_cols = [f'{ch}_SPEND' for ch in top_channels]

# Daily aggregation for pairplot
pairplot_data = df.groupby('DATE_DAY')[top_spend_cols + ['revenue']].sum().reset_index()
pairplot_data.columns = ['DATE_DAY'] + [c.replace('_SPEND', '') for c in top_spend_cols] + ['Revenue']

# Create pairplot
g = sns.pairplot(
    pairplot_data.drop(columns='DATE_DAY'),
    diag_kind='kde',
    plot_kws={'alpha': 0.5, 's': 30},
    corner=True
)
g.fig.suptitle('Bivariate Relationships: Top 3 Spend Channels vs Revenue', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Scatterplots with regression lines
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, channel in enumerate(top_channels):
    spend_col = f'{channel}_SPEND'
    channel_daily = df.groupby('DATE_DAY')[[spend_col, 'revenue']].sum().reset_index()
    
    sns.regplot(
        data=channel_daily,
        x=spend_col,
        y='revenue',
        ax=axes[i],
        scatter_kws={'alpha': 0.5},
        line_kws={'color': 'red'}
    )
    
    # Calculate correlation
    corr = channel_daily[spend_col].corr(channel_daily['revenue'])
    axes[i].set_title(f'{channel}\nr = {corr:.3f}')
    axes[i].set_xlabel('Daily Spend')
    axes[i].set_ylabel('Daily Revenue')
    axes[i].xaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x/1e3:.0f}K'))
    axes[i].yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

plt.suptitle('Spend vs Revenue by Top Channels', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 9.2 Lagged Correlation Analysis (Adstock Effect)

In [ ]:
# Lagged correlations to capture delayed marketing effects
if 'revenue' in df.columns and spend_cols:
    daily_agg = df.groupby('DATE_DAY')[spend_cols + ['revenue']].sum().reset_index()
    daily_agg = daily_agg.sort_values('DATE_DAY')
    
    lags = [0, 1, 7, 14, 30]
    lag_corr_results = []
    
    for lag in lags:
        lagged_revenue = daily_agg['revenue'].shift(-lag)
        for spend_col in spend_cols:
            corr = daily_agg[spend_col].corr(lagged_revenue)
            lag_corr_results.append({
                'Channel': spend_col.replace('_SPEND', ''),
                'Lag (days)': lag,
                'Correlation': corr
            })
    
    lag_corr_df = pd.DataFrame(lag_corr_results)
    lag_corr_pivot = lag_corr_df.pivot(index='Channel', columns='Lag (days)', values='Correlation')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(lag_corr_pivot, annot=True, cmap='RdYlGn', center=0, ax=ax, fmt='.2f')
    ax.set_title('Lagged Correlation: Spend (t) vs Revenue (t + lag)')
    ax.set_xlabel('Lag (days)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Identify optimal lag for each channel
if 'revenue' in df.columns and spend_cols:
    optimal_lags = lag_corr_df.loc[lag_corr_df.groupby('Channel')['Correlation'].idxmax()]
    print("Optimal lag (highest correlation) per channel:")
    display(optimal_lags.sort_values('Correlation', ascending=False))

## 10. Feature Engineering Insights

This section summarizes transformations and features that may improve MMM model performance.

### 10.1 Recommended Transformations

| Transformation | Reason | Target Variables |
|----------------|--------|------------------|
| **Log(x + 1)** | Revenue and spend are right-skewed | Revenue, all spend columns |
| **Adstock** | Marketing effects decay over time | All spend columns |
| **Saturation** | Diminishing returns at higher spend | All spend columns |

In [ ]:
# Log transformation example
print("Skewness before transformation:")
for col in spend_cols[:3] + ['revenue']:
    print(f"  {col}: {df[col].skew():.2f}")

print("\nSkewness after log(x+1) transformation:")
for col in spend_cols[:3] + ['revenue']:
    print(f"  {col}: {np.log1p(df[col]).skew():.2f}")

### 10.2 Suggested Lag Features

Based on the lagged correlation analysis:

In [ ]:
# Display optimal lags for feature engineering
if 'lag_corr_df' in dir():
    print("Recommended lag features per channel:")
    for _, row in optimal_lags.iterrows():
        print(f"  {row['Channel']}_SPEND_LAG{int(row['Lag (days)'])}d (r = {row['Correlation']:.3f})")

### 10.3 Calendar Features

The following calendar features should be added to the preprocessing pipeline:

- **day_of_week**: Already computed; shows weekend effects
- **month**: Captures monthly seasonality  
- **is_holiday**: Already computed; significant external regressor
- **is_weekend**: Saturday/Sunday indicator
- **quarter**: Q4 typically has higher revenue (holiday shopping)

In [ ]:
# Calendar features example
df['is_weekend'] = df['DATE_DAY'].dt.dayofweek >= 5
df['month'] = df['DATE_DAY'].dt.month
df['quarter'] = df['DATE_DAY'].dt.quarter

print("Calendar feature distribution:")
print(f"  Weekend days: {df['is_weekend'].sum():,} ({df['is_weekend'].mean()*100:.1f}%)")
print(f"  Holiday days: {df['is_holiday'].sum():,} ({df['is_holiday'].mean()*100:.1f}%)")

# Average revenue by quarter
print("\nAverage revenue by quarter:")
print(df.groupby('quarter')['revenue'].mean().apply(lambda x: f'${x:,.0f}'))

### 10.4 Interaction Terms

Potential interaction terms for the MMM model:

- **spend × region**: Different ROI by market
- **spend × is_holiday**: Holiday amplification effects
- **spend × quarter**: Seasonal spend effectiveness

## 11. Key Findings Summary

In [ ]:
print("=" * 60)
print("EDA SUMMARY")
print("=" * 60)
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['DATE_DAY'].min().date()} to {df['DATE_DAY'].max().date()}")
print(f"Unique organisations: {df['ORGANISATION_ID'].nunique()}")
print(f"Unique regions: {df['ORGANISATION_PRIMARY_TERRITORY_NAME'].nunique()}")
print(f"\nSpend channels: {[c.replace('_SPEND', '') for c in spend_cols]}")
if 'revenue' in df.columns:
    print(f"\nTotal revenue: ${df['revenue'].sum():,.0f}")
    print(f"Mean daily revenue: ${daily['revenue'].mean():,.0f}")
print(f"\nMissing data (top 5):")
print(missing.head())
print("=" * 60)

### Key Insights

1. **Missing Data**: TikTok channel has highest missing rates; consider imputation strategy
2. **Revenue Distribution**: Highly skewed; log transformation recommended
3. **Temporal Patterns**: Clear weekly seasonality; holidays show different behavior
4. **Channel Efficiency**: Google Paid Search has highest CTR; TikTok has lowest CPC
5. **Regional Variation**: Significant revenue concentration in specific regions
6. **Adstock Effects**: Different channels show optimal lags ranging from 0-30 days